# M2aSoiling — Analysis Run (SRR rdtools)

Prasyarat di Google Drive `Cek PV String/`:
- `baseline/` — CSV baseline `{YYYY-MM}/{YYYY-MM-DD}.csv`
- `raw data input/` — Daily Rainfall PLTS IKN 2025/2026.xlsx, Report & Schedule Cleaning PLTS IKN.xlsx, List of DC Cables 0411.xls, **POA PLTS IKN 2025/2026.xlsx**, **Surface Albedo Forecast TMY NSRDB PLTS IKN.xlsx**, **PV Module Temperature PLTS IKN.xlsx** (untuk koreksi suhu SRR)

Tanpa file POA/albedo, POA jatuh ke pvlib clearsky → PR harian berisik → rawan `NoValidIntervalError`.

Jalankan cell berurut. Tiap varian analisis (WB03-10 / WB01-02) dipanggil lewat helper `run_soiling()` — argumen dikirim sebagai list ke subprocess, jadi path berspasi & pengulangan argumen aman.

> Analisis blended satu-site sengaja **tidak** disertakan: menggabung dua zona dengan kadens cleaning berbeda membuat deret PR terpecah menjadi interval < min_interval_length, sehingga SRR menghasilkan `NoValidIntervalError`. Soiling dianalisis per zona cleaning.

**Output tiap file** kini memuat sheet `CleaningImpact`: per event cleaning -> `sr_before`, `sr_after`, `uplift_pct`, `energy_recovered_kwh_per_day`, `rupiah_per_day`, `likely_cause`.

Cell terakhir menjalankan **per-WB tunggal (WB01..WB10)** sekaligus (baseline dimuat sekali; kapasitas & biaya per-WB dari tabel bawaan).

In [ ]:
# Cell 1 - Mount Drive + FRESH clone (cegah clone bersarang saat re-run)
from google.colab import drive
drive.mount("/content/drive")

%cd /content
!rm -rf PVStringHeatmapCheck
!git clone https://github.com/ompltsikn/PVStringHeatmapCheck.git
%cd /content/PVStringHeatmapCheck

In [ ]:
# Cell 2 - Salin POA + albedo dari Drive ke "raw data input/" repo.
# POAProvider membacanya relatif terhadap cwd; tanpa ini POA fallback
# ke pvlib clearsky (PR berisik).
BASE = "/content/drive/MyDrive/Cek PV String"   # sesuaikan dengan path Drive Anda

!mkdir -p "raw data input"
!cp "{BASE}/raw data input/POA"*.xlsx "raw data input/" || echo "WARNING: file POA tidak ada di Drive -> POA pakai clearsky"
!cp "{BASE}/raw data input/Surface Albedo"*.xlsx "raw data input/" || echo "WARNING: file albedo tidak ada di Drive -> albedo statis 0.2"
!cp "{BASE}/raw data input/PV Module Temperature"*.xlsx "raw data input/" || echo "WARNING: file Tcell tidak ada -> koreksi suhu OFF (CF=1)"
!ls -la "raw data input/"

In [ ]:
# Cell 3 - Helper run_soiling(): argumen dikirim sebagai LIST ke subprocess.
# Tanpa shell -> tak ada masalah quoting path berspasi / line-continuation.
import subprocess
import sys


def run_soiling(*, cleaning_cost_idr=None, wb=None, capacity_kwp=None, per_wb=False):
    cmd = [
        sys.executable, "-u", "run_soiling_analysis.py",
        "--baseline-dir", f"{BASE}/baseline",
        "--rainfall-xlsx",
        f"{BASE}/raw data input/Daily Rainfall PLTS IKN 2025.xlsx",
        f"{BASE}/raw data input/Daily Rainfall PLTS IKN 2026.xlsx",
        "--cleaning-report-xlsx",
        f"{BASE}/raw data input/Report & Schedule Cleaning PLTS IKN.xlsx",
        "--dc-cable-xls", f"{BASE}/raw data input/List of DC Cables 0411.xls",
        "--clean-criterion", "precip_and_shift",
        "--precip-threshold-mm", "1.0",
        "--min-interval-length", "5",
        "--output-dir", f"{BASE}/outputs",
    ]
    if per_wb:
        cmd += ["--per-wb"]
    else:
        cmd += ["--cleaning-cost-idr", str(cleaning_cost_idr)]
        if wb:
            cmd += ["--wb", *wb]
        if capacity_kwp is not None:
            cmd += ["--capacity-kwp", str(capacity_kwp)]
    print(">>", " ".join(cmd))
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print("[exit code]", rc)
    return rc

In [ ]:
# Cell 4 - Kelompok WB03-10 (water truck solar; kapasitas ~58.000 kWp)
run_soiling(
    wb=["WB03", "WB04", "WB05", "WB06", "WB07", "WB08", "WB09", "WB10"],
    capacity_kwp=58000,
    cleaning_cost_idr=55000000,
)

In [ ]:
# Cell 5 - Kelompok WB01-02 (gravitasi + pompa; kapasitas ~13.500 kWp)
run_soiling(
    wb=["WB01", "WB02"],
    capacity_kwp=13500,
    cleaning_cost_idr=1165000,
)

In [ ]:
# Cell 6 - Per-WB tunggal WB01..WB10 (baseline dimuat SEKALI).
# Kapasitas & cleaning cost per WB dari tabel bawaan run_soiling_analysis.py
# (PER_WB_CAPACITY_KWP / PER_WB_CLEANING_COST_IDR). Menghasilkan 10 file:
# soiling_srr_<periode>_WB01.xlsx .. _WB10.xlsx, masing-masing dengan sheet
# CleaningImpact (uplift kWh/rupiah dipulihkan per event cleaning).
run_soiling(per_wb=True)